In [1]:
from datetime import datetime
from typing import Literal, Optional
import pybamm
import pandas as pd
import plotly.graph_objects as go
from netlist_utils import setup_circuit
from battery import LiionBatteryPack, PybammBattery, CLCBattery
import vessim as vs

import nest_asyncio
nest_asyncio.apply()

In [2]:
class MockActor(vs.Actor):
    def __init__(self, name: str = "mock", step_size = None, p = 0):
        super().__init__(name, step_size)
        self.power = p

    def p(self, now: datetime) -> float:
        return self.power

In [3]:
class CarbonAwareController(vs.Controller):
    def __init__(self, power_meters: list[vs.MockPowerMeter], ci: vs.HistoricalSignal):
        super().__init__()
        self.power_meters = power_meters
        self.ci = ci

    def step(self, time: datetime, p_delta: float, e_delta: float, state: dict) -> None:
        self.set_parameters["policy:min_soc"] = 0.5 if (time.hour < 11 or time.hour > 13) and state["storage"]["soc"] >= 0.5 else 0.3
        self.set_parameters["policy:charge_power"] = 40 if (time.hour <= 4 or time.hour >= 23) and self.ci.at(time) <= 60 else 0
        self.power_meters[0].set_power(9 if 3 <= time.hour < 9 else 18)

In [4]:
class MinMaxSocPolicy(vs.MicrogridPolicy):
    def __init__(self, mode: Literal["grid-connected", "islanded"] = "grid-connected", charge_power: Optional[float] = None, min_soc = 0, max_soc=1):
        self.mode = mode
        self.charge_power = charge_power if charge_power else 0.0
        self.min_soc = min_soc
        self.max_soc = max_soc

    def apply(self, p_delta: float, duration: int, storage: Optional[vs.Storage] = None) -> float:
        energy_delta = p_delta * duration
        if storage and ((self.min_soc >= storage.soc() and p_delta <= 0) or (self.max_soc <= storage.soc() and p_delta >= 0)):
            if self.mode == "islanded":
                if energy_delta < 0.0:
                    raise RuntimeError("Not enough energy available to operate in islanded mode.")
                energy_delta = 0.0
        elif self.mode == "grid-connected" and storage is not None:
            if self.charge_power:
                energy_delta -= storage.update(self.charge_power, duration)
            else:
                energy_delta -= storage.update(p_delta, duration)
        elif self.mode == "islanded":
            if storage:
                energy_delta -= storage.update(p_delta, duration)
            if energy_delta < 0.0:
                raise RuntimeError("Not enough energy available to operate in islanded mode.")
            energy_delta = 0.0
        return energy_delta

    def state(self) -> dict:
        return {
            "mode": self.mode,
            "charge_power": self.charge_power,
            "min_soc": self.min_soc
        }


In [8]:
def init_battery_models(initial_soc, np, ns, step_size):
    simple_battery = vs.SimpleBattery(capacity=18.8 * np * ns, initial_soc=initial_soc)
    clc_battery = CLCBattery(number_of_cells=ns*np, initial_soc=initial_soc)
    pybamm_battery = PybammBattery(
        model=pybamm.lithium_ion.SPMe(options={"operating mode": "power", "calculate discharge energy": "true"}),
        initial_soc=initial_soc,
        number_of_cells=ns*np,
        parameter_values=pybamm.ParameterValues("Chen2020"),
        output_variables=["Current [A]", "Voltage [V]", "Power [W]", "Discharge energy [W.h]"],
    )
    liion_battery_pack = LiionBatteryPack(
        model = pybamm.lithium_ion.SPMe(),
        netlist = setup_circuit(Np=np, Ns=ns, Rb=1.5e-3, Rc=1e-2, Ri=5e-2, V=4.0, I=5.0),
        step_size=step_size,
        initial_soc=initial_soc,
        parameter_values=pybamm.ParameterValues("Chen2020"),
    )
    return [simple_battery, clc_battery, pybamm_battery, liion_battery_pack]

In [ ]:
STEP_SIZE = 60
environment = vs.Environment(sim_start="2022-07-03 09:00:00")
for battery in init_battery_models(initial_soc=0.7, np=4, ns=4, step_size=STEP_SIZE):
    power_meters: list = [
        vs.MockPowerMeter(name="jetson", p=18),
        # MockPowerMeter(name="virtual", p=32)
    ]
    policy = MinMaxSocPolicy(min_soc=0.5)
    ci = vs.HistoricalSignal(actual=pd.read_csv("data/watttime2022_caiso-north_actual.csv", index_col=0))
    monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}.csv", grid_signals={"carbon-intensity": ci})
    ca_controller = CarbonAwareController(power_meters=power_meters, ci=ci)
    environment.add_microgrid(
        actors=[
            vs.ComputingSystem(power_meters=power_meters),
            vs.Generator(
                signal=vs.HistoricalSignal.from_dataset("solcast2022_global", params={"scale": 60}),
                column="San Francisco",
            ),
        ],
        policy=policy,
        storage=battery,
        controllers=[monitor, ca_controller],
        step_size=STEP_SIZE,  # Global step size (can be overridden by actors or controllers)
    )

environment.run(until=24 * 3600)

In [ ]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for battery in ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]:
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    fig.add_trace(
        go.Scatter(x=[i * STEP_SIZE for i in range(len(df.index))], y=df["storage.soc"], name=battery, mode="lines")
    )
fig.show()

In [9]:
STEP_SIZE = 10
POWER = -19.05 / 5 * 16
environment = vs.Environment(sim_start="2022-07-03 09:00:00")
for battery in init_battery_models(initial_soc=1, np=4, ns=4, step_size=STEP_SIZE):
    monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}.csv")
    environment.add_microgrid(
        actors=[MockActor(p=POWER)],
        storage=battery,
        controllers=[monitor],
        step_size=STEP_SIZE,
    )

environment.run(until=5 * 3600)

2024-06-20 13:55:17.923 | INFO     | mosaik.scenario:start:280 - Starting "Actor" as "Actor-0" ...
2024-06-20 13:55:17.924 | INFO     | mosaik.scenario:start:280 - Starting "Grid" as "Grid-0" ...
2024-06-20 13:55:17.924 | INFO     | mosaik.scenario:start:280 - Starting "Controller" as "Controller-0" ...
2024-06-20 13:55:17.925 | INFO     | mosaik.scenario:start:280 - Starting "Storage" as "Storage-0" ...
2024-06-20 13:55:17.925 | INFO     | mosaik.scenario:start:280 - Starting "Actor" as "Actor-1" ...
2024-06-20 13:55:17.926 | INFO     | mosaik.scenario:start:280 - Starting "Grid" as "Grid-1" ...
2024-06-20 13:55:17.926 | INFO     | mosaik.scenario:start:280 - Starting "Controller" as "Controller-1" ...
2024-06-20 13:55:17.927 | INFO     | mosaik.scenario:start:280 - Starting "Storage" as "Storage-1" ...
2024-06-20 13:55:17.927 | INFO     | mosaik.scenario:start:280 - Starting "Actor" as "Actor-2" ...
2024-06-20 13:55:17.928 | INFO     | mosaik.scenario:start:280 - Starting "Grid" as "

In [19]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=300,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend={"xanchor": "right", "x": 1}
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{Time}$", ticksuffix="h", range=[0, 5.07]
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", ticks="outside", title="$\\text{State-of-Charge}$", ticksuffix="%", range=[0, 103]
)
for battery in ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]:
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    fig.add_trace(
        go.Scatter(x=[i * STEP_SIZE / 3600 for i in range(len(df.index))], y=df["storage.soc"] * 100, name=battery, mode="lines")
    )
fig.write_image("../thesis/Images/discharge_energy.pdf", "pdf")

In [ ]:
STEP_SIZE = 10
environment = vs.Environment(sim_start="2022-07-03 09:00:00")
for battery in init_battery_models(initial_soc=0, np=4, ns=4, step_size=STEP_SIZE):
    monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}.csv")
    environment.add_microgrid(
        actors=[MockActor(p=120)],
        storage=battery,
        controllers=[monitor],
        step_size=STEP_SIZE,
    )

environment.run(until=4 * 3600)

In [ ]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for battery in ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]:
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    fig.add_trace(
        go.Scatter(x=[i * STEP_SIZE for i in range(len(df.index))], y=df["storage.soc"], name=battery, mode="lines")
    )
fig.show()

In [ ]:
STEP_SIZE = 5
POWER = -7.5
pybamm_battery = PybammBattery(
    model=pybamm.lithium_ion.SPMe(options={"operating mode": "power", "calculate discharge energy": "true"}),
    initial_soc=1,
    number_of_cells=1,
    parameter_values=pybamm.ParameterValues("Chen2020"),
    output_variables=["Current [A]", "Voltage [V]", "Power [W]", "Discharge energy [W.h]"],
)
environment = vs.Environment(sim_start="2022-07-03 09:00:00")
monitor = vs.Monitor(outfile=f"results/{type(pybamm_battery).__name__}.csv")
environment.add_microgrid(
    actors=[MockActor(p=POWER)],
    storage=pybamm_battery,
    controllers=[monitor],
    step_size=STEP_SIZE,
)

environment.run(until=5 * 3600)

In [ ]:
df = pd.read_csv("results/PybammBattery.csv", index_col = 0)
df = df[df["e_delta"] == 0.0]

print((df["storage.single_cell_Power [W]"].sum() - POWER) * STEP_SIZE / 3600)
print((df["storage.single_cell_Power [W]"].sum()) * STEP_SIZE / 3600)
print(df["p_delta"].sum() * STEP_SIZE / 3600)